# Fase 4 — Salida: PDF + PostgreSQL

**Pipeline DIGI Spain Telecom — Control de Calidad C2C**  
**Nodos:** 12 (Informe PDF) · 13 (PostgreSQL)

> Genera el **Informe de Rendimiento de Auditoria** en formato real DIGI:
> Pagina 1 — Resumen de Rendimiento
> Pagina 2 — Estructura de la llamada
> Pagina 3 — Actitud / Comunicacion + Feedback Adicional (orientado al agente)


## PASO 0 — Auto-fix NumPy (ejecutar siempre primero)

In [ ]:
import subprocess, sys, importlib.util
spec = importlib.util.find_spec('numpy')
if spec:
    import numpy as _np
    ver = tuple(int(x) for x in _np.__version__.split('.')[:2])
    if ver >= (2, 0):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2'])
        print('Reiniciar runtime y re-ejecutar')
    else:
        print(f'NumPy {_np.__version__} OK')
else:
    print('NumPy no instalado aun')


## PASO 1 — Instalar dependencias

In [ ]:
get_ipython().system('pip install -q reportlab psycopg2-binary')
get_ipython().system('apt-get install -q -y fonts-dejavu > /dev/null 2>&1')
print('Dependencias listas')


## PASO 2 — Configuracion

In [ ]:
import json
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR       = Path('/content/drive/MyDrive/auditoria-digi')
RESULTADO_PATH = BASE_DIR / 'outputs' / 'resultado_fase3.json'
PDF_PATH       = BASE_DIR / 'outputs' / 'informe_auditoria_digi.pdf'

print(f'BASE_DIR : {BASE_DIR}')
print(f'Input    : {RESULTADO_PATH}')
print(f'PDF out  : {PDF_PATH}')


## PASO 3 — Cargar resultado_fase3.json

In [ ]:
with open(RESULTADO_PATH, 'r', encoding='utf-8') as f:
    resultado = json.load(f)

crit = resultado.get('criterios', [])
nota = resultado.get('nota_final', 0.0)
print('JSON cargado')
print(f'   Asesor      : {resultado.get("asesor", "N/D")}')
print(f'   ID grabacion: {resultado.get("id_grabacion", "N/D")}')
print(f'   Criterios   : {len(crit)}')
print(f'   Nota final  : {nota:.2f}')


## NODO 12a — Estilos y paleta DIGI

> Paleta oficial: azul #003087, naranja #FF6600.  
> Fuentes DejaVu para soporte UTF-8 (n, a, e, u).  
> Columnas tabla: Pregunta · Respuesta · Puntuacion · Comentarios (sin columna ID).


In [ ]:
from collections import OrderedDict
import re
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.platypus import (
    BaseDocTemplate, PageTemplate, Frame,
    Table, TableStyle, Paragraph, Spacer, PageBreak,
    KeepTogether,
)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfgen import canvas as rl_canvas

# Fuentes DejaVu
try:
    pdfmetrics.registerFont(TTFont('DJ',  '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'))
    pdfmetrics.registerFont(TTFont('DJB', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'))
    pdfmetrics.registerFont(TTFont('DJI', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Oblique.ttf'))
    FN, FB, FI = 'DJ', 'DJB', 'DJI'
    print('Fuentes DejaVu OK')
except Exception as e:
    FN, FB, FI = 'Helvetica', 'Helvetica-Bold', 'Helvetica-Oblique'
    print(f'Helvetica (fallback): {e}')

# Paleta DIGI
C_BLUE     = colors.HexColor('#003087')
C_BLUE_MID = colors.HexColor('#1F5FAD')
C_BLUE_L   = colors.HexColor('#C5D9F1')
C_BLUE_VL  = colors.HexColor('#EEF4FB')
C_ORANGE   = colors.HexColor('#FF6600')
C_SI       = colors.HexColor('#1A7C36')
C_NO       = colors.HexColor('#C0392B')
C_NO_BG    = colors.HexColor('#FDECEA')
C_NA       = colors.HexColor('#7F8C8D')
C_WHITE    = colors.white
C_DARK     = colors.HexColor('#1C2B3A')
C_BORDER   = colors.HexColor('#C0C8D4')
C_LGRAY    = colors.HexColor('#F2F4F6')

W, H = A4
MARGIN = 35
TW = W - 2 * MARGIN

# Anchos de columna tabla criterios
COL_PREG  = 238
COL_RESP  = 57
COL_PUNT  = 65
COL_COM   = TW - COL_PREG - COL_RESP - COL_PUNT
COL_WIDTHS = [COL_PREG, COL_RESP, COL_PUNT, COL_COM]

# Estilos
def _s(name, **kw):
    d = dict(fontName=FN, fontSize=9, leading=11, textColor=C_DARK)
    d.update(kw)
    return ParagraphStyle(name, **d)

S_SUBTITLE = _s('subtitle', fontName=FB, fontSize=11, textColor=C_BLUE, leading=14)
S_INFO_L   = _s('info_l',   fontSize=9,  leading=13)
S_NOTA_BIG = _s('nota_big', fontName=FB, fontSize=30, textColor=C_ORANGE,
                 alignment=TA_CENTER, leading=34)
S_NOTA_LBL = _s('nota_lbl', fontName=FB, fontSize=9,  textColor=C_BLUE,
                 alignment=TA_CENTER, leading=12)
S_COL_HDR  = _s('col_hdr',  fontName=FB, fontSize=8.5, textColor=C_WHITE,
                 alignment=TA_CENTER, leading=11)
S_SEC_HDR  = _s('sec_hdr',  fontName=FB, fontSize=9.5, textColor=C_WHITE, leading=12)
S_SUB_HDR  = _s('sub_hdr',  fontName=FB, fontSize=9,   textColor=C_DARK,  leading=11)
S_PREGUNTA = _s('pregunta', fontSize=8,  leading=10.5, textColor=C_DARK)
S_COMENT   = _s('coment',   fontSize=7.5, leading=10,
                 textColor=colors.HexColor('#555555'))
S_FBK_BLD  = _s('fbk_bld',  fontName=FB, fontSize=9,  leading=12, textColor=C_DARK)
S_FBK_TXT  = _s('fbk_txt',  fontSize=9,  leading=13,  textColor=C_DARK)

def _nota_color(n):
    if n >= 7.5: return C_SI
    if n >= 5.0: return colors.HexColor('#D4820E')
    return C_NO

def _fmt(n):
    return f'{n:.2f}'.replace('.', ',')

def _fmt_punt(n):
    return f'{n:.2f}'.replace('.', ',')

print(f'Estilos DIGI | A4: {round(W)}x{round(H)} pt | TW: {round(TW)} pt')


## NODO 12b — Agrupacion de criterios por seccion

> Mapea cada criterio a su seccion (Presentacion, Politica de seguridad...)
> y grupo (Estructura de la llamada / Actitud / Comunicacion).


In [ ]:
SECCION_POR_ID = {
    'ps': 'Presentación',
    'se': 'Política de seguridad',
    'so': 'Sondeo',
    'ao': 'Asignación de oferta',
    'cl': 'Cierre de la venta',
    'td': 'Toma de datos y registro en sistemas',
    'cd': 'Cierre de la llamada',
    'tc': 'Trato al cliente',
    'ge': 'Gestión de esperas',
    'cc': 'Capacidad de comunicación',
}

SECCIONES_ACTITUD = {
    'Trato al cliente', 'Gestión de esperas', 'Capacidad de comunicación'
}

ORDEN_ESTRUCTURA = [
    'Presentación', 'Política de seguridad', 'Sondeo',
    'Asignación de oferta', 'Cierre de la venta',
    'Toma de datos y registro en sistemas', 'Cierre de la llamada',
]
ORDEN_ACTITUD = [
    'Trato al cliente', 'Gestión de esperas', 'Capacidad de comunicación'
]

def _inferir_seccion(cid):
    m = re.match(r'^([a-z]+)\d+$', str(cid))
    return SECCION_POR_ID.get(m.group(1), 'Otro') if m else 'Otro'

def _inferir_grupo(sec):
    return 'Actitud / Comunicación' if sec in SECCIONES_ACTITUD else 'Estructura de la llamada'

def agrupar_criterios(criterios):
    grupos = {
        'Estructura de la llamada': OrderedDict(),
        'Actitud / Comunicación':   OrderedDict(),
    }
    for c in criterios:
        sec = (c.get('seccion') or c.get('subseccion') or _inferir_seccion(c.get('id','')))
        grp = (c.get('seccion_grupo') or c.get('grupo') or _inferir_grupo(sec))
        if grp not in grupos:
            grp = 'Estructura de la llamada'
        if sec not in grupos[grp]:
            grupos[grp][sec] = []
        grupos[grp][sec].append(c)
    for grp, orden in [('Estructura de la llamada', ORDEN_ESTRUCTURA),
                       ('Actitud / Comunicación', ORDEN_ACTITUD)]:
        reord = OrderedDict()
        for s in orden:
            if s in grupos[grp]:
                reord[s] = grupos[grp][s]
        for s in grupos[grp]:
            if s not in reord:
                reord[s] = grupos[grp][s]
        grupos[grp] = reord
    return grupos

print('agrupar_criterios() lista')


## NODO 12c — Constructor de tabla de criterios

> Formato real DIGI:
> - Fila de grupo (dark blue, span 4 cols)
> - Cabecera Pregunta / Respuesta / Puntuacion / Comentarios (medium blue)
> - Filas de subseccion (azul claro, span 4 cols)
> - Filas de datos: No en fondo rojo (visible para el agente), N/A en gris


In [ ]:
def tabla_criterios(grupo_nombre, secciones_dict):
    data   = []
    styles = []

    # Fila 0: nombre del grupo (azul DIGI, texto blanco)
    data.append([Paragraph(grupo_nombre, S_SEC_HDR), '', '', ''])
    styles += [
        ('SPAN',         (0,0), (-1,0)),
        ('BACKGROUND',   (0,0), (-1,0), C_BLUE),
        ('TOPPADDING',   (0,0), (-1,0), 7),
        ('BOTTOMPADDING',(0,0), (-1,0), 7),
        ('LEFTPADDING',  (0,0), (-1,0), 10),
    ]

    # Fila 1: cabeceras de columna (azul medio, texto blanco)
    data.append([
        Paragraph('Pregunta',    S_COL_HDR),
        Paragraph('Respuesta',   S_COL_HDR),
        Paragraph('Puntuación',  S_COL_HDR),
        Paragraph('Comentarios', S_COL_HDR),
    ])
    styles += [
        ('BACKGROUND',   (0,1), (-1,1), C_BLUE_MID),
        ('TOPPADDING',   (0,1), (-1,1), 5),
        ('BOTTOMPADDING',(0,1), (-1,1), 5),
    ]

    for sec_name, items in secciones_dict.items():
        # Fila de subseccion (azul claro, full width)
        r = len(data)
        data.append([Paragraph(sec_name, S_SUB_HDR), '', '', ''])
        styles += [
            ('SPAN',         (0,r), (-1,r)),
            ('BACKGROUND',   (0,r), (-1,r), C_BLUE_L),
            ('TOPPADDING',   (0,r), (-1,r), 4),
            ('BOTTOMPADDING',(0,r), (-1,r), 4),
            ('LEFTPADDING',  (0,r), (-1,r), 12),
        ]

        for item in items:
            resp    = item.get('respuesta', 'N/A')
            punt    = item.get('puntuacion', 0.0)
            desc    = (item.get('descripcion') or item.get('pregunta') or '')
            comment = item.get('comentario', '') or ''
            critico = item.get('critico', False) or item.get('critica', False)

            if critico and not desc.endswith('*'):
                desc = desc + '*'

            resp_norm = 'Sí' if resp in ('Sí', 'Si', 'SÍ', 'SI', 'sí', 'si') else resp
            if resp_norm == 'Sí':
                resp_color, row_bg = C_SI,  None
            elif resp_norm == 'No':
                resp_color, row_bg = C_NO,  C_NO_BG
            else:
                resp_color, row_bg = C_NA,  None
                resp_norm = 'N/A'

            punt_str = '0' if resp_norm == 'N/A' else _fmt_punt(punt)

            r = len(data)
            data.append([
                Paragraph(desc, S_PREGUNTA),
                Paragraph(resp_norm,
                          ParagraphStyle(f'rsp{r}', parent=_s(f'r{r}'),
                                         fontName=FB, fontSize=8.5,
                                         alignment=TA_CENTER, textColor=resp_color,
                                         leading=10.5)),
                Paragraph(punt_str,
                          ParagraphStyle(f'pnt{r}', parent=_s(f'p{r}'),
                                         fontSize=8.5, alignment=TA_CENTER,
                                         leading=10.5)),
                Paragraph(comment, S_COMENT),
            ])
            styles += [
                ('TOPPADDING',   (0,r), (-1,r), 3),
                ('BOTTOMPADDING',(0,r), (-1,r), 3),
                ('LEFTPADDING',  (0,r), (-1,r), 6),
                ('RIGHTPADDING', (0,r), (-1,r), 4),
            ]
            if row_bg:
                styles.append(('BACKGROUND', (0,r), (-1,r), row_bg))

    # Estilos globales
    styles += [
        ('FONT',         (0,0), (-1,-1), FN, 8),
        ('VALIGN',       (0,0), (-1,-1), 'MIDDLE'),
        ('GRID',         (0,0), (-1,-1), 0.25, C_BORDER),
        ('LEFTPADDING',  (0,0), (-1,-1), 4),
        ('RIGHTPADDING', (0,0), (-1,-1), 4),
    ]

    return Table(data, colWidths=COL_WIDTHS, repeatRows=2,
                 style=TableStyle(styles), hAlign='LEFT')

print('tabla_criterios() lista')


## NODO 12d — Canvas DIGI + construccion de paginas

> **Pagina 1:** Resumen de Rendimiento (asesor, nota grande, tabla auditoria, estadisticas)
> **Pagina 2:** Ficha de auditoria + Tabla Estructura de la llamada
> **Pagina 3:** Tabla Actitud / Comunicacion + Feedback Adicional
> (Observaciones narrativas + Puntos Fuertes + Areas de Mejora por seccion)


In [ ]:
# Canvas DIGI con header y footer en cada pagina
class DIGICanvas(rl_canvas.Canvas):
    def __init__(self, *args, **kwargs):
        self._res = kwargs.pop('resultado', {})
        self._pg_states = []
        super().__init__(*args, **kwargs)

    def showPage(self):
        self._pg_states.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        n = len(self._pg_states)
        for state in self._pg_states:
            self.__dict__.update(state)
            self._deco(n)
            rl_canvas.Canvas.showPage(self)
        rl_canvas.Canvas.save(self)

    def _deco(self, n_pages):
        self.saveState()
        pg = self._pageNumber
        # Barra azul header
        self.setFillColor(C_BLUE)
        self.rect(0, H - 30, W, 30, fill=1, stroke=0)
        # Franja naranja
        self.setFillColor(C_ORANGE)
        self.rect(0, H - 34, W, 4, fill=1, stroke=0)
        # Titulo
        self.setFillColor(C_WHITE)
        self.setFont(FB, 10)
        self.drawCentredString(W / 2, H - 20, 'Informe de Rendimiento de Auditoría')
        # Pie
        self.setFillColor(C_LGRAY)
        self.rect(0, 0, W, 22, fill=1, stroke=0)
        self.setFillColor(C_NA)
        self.setFont(FN, 7)
        asesor  = self._res.get('asesor', '')
        fecha_a = self._res.get('fecha_auditoria', '')
        self.drawString(MARGIN, 7,
            f'Asesor: {asesor}  |  Fecha auditoria: {fecha_a}  |  DIGI Spain Telecom - C2C - CONFIDENCIAL')
        self.drawRightString(W - MARGIN, 7, f'Pagina {pg}')
        self.restoreState()


# Helpers
def _fmt_list(items, fallback='Sin datos.'):
    if not items:
        return fallback
    lines = [f'• {x}' for x in items if x and str(x).strip()]
    return '<br/>'.join(lines) if lines else fallback

def _fmt_areas(items, fallback='Sin areas de mejora.'):
    if not items:
        return fallback
    lines = []
    for x in items:
        if isinstance(x, dict):
            sec = x.get('seccion', '')
            txt = x.get('descripcion', x.get('texto', ''))
            lines.append(f'<b>{sec}:</b> {txt}' if sec else txt)
        elif x and str(x).strip():
            lines.append(str(x))
    return '<br/>'.join(lines) if lines else fallback


# PAGINA 1 — Resumen de Rendimiento
def build_resumen(res):
    asesor   = res.get('asesor', 'N/D')
    periodo  = res.get('periodo') or res.get('fecha_llamada', 'N/D')
    servicio = res.get('servicio', 'C2C')
    nota     = res.get('nota_final', 0.0)
    id_grab  = res.get('id_grabacion', 'N/D')
    crit_all = res.get('criterios', [])

    story = []
    story.append(Spacer(1, 4))
    story.append(Paragraph('Resumen de Rendimiento', S_SUBTITLE))
    story.append(Spacer(1, 8))

    nc = _nota_color(nota)
    info_rows = [
        [Paragraph(f'<b>Asesor:</b> {asesor}', S_INFO_L),
         Paragraph('Nota Media del Periodo:', S_NOTA_LBL)],
        [Paragraph(f'<b>Periodo:</b> {periodo}', S_INFO_L),
         Paragraph(_fmt(nota), ParagraphStyle('nb', parent=S_NOTA_BIG, textColor=nc))],
        [Paragraph(f'<b>Servicio:</b> {servicio}', S_INFO_L), ''],
    ]
    it = Table(info_rows, colWidths=[TW * 0.65, TW * 0.35])
    it.setStyle(TableStyle([
        ('SPAN',          (1,1),(1,2)),
        ('VALIGN',        (0,0),(-1,-1), 'MIDDLE'),
        ('ALIGN',         (1,0),(-1,-1), 'CENTER'),
        ('LINEBELOW',     (0,2),(-1,2), 0.5, C_BORDER),
        ('TOPPADDING',    (0,0),(-1,-1), 4),
        ('BOTTOMPADDING', (0,0),(-1,-1), 4),
    ]))
    story.append(it)
    story.append(Spacer(1, 12))

    # Tabla detalle auditorias
    audit_data = [
        [Paragraph('<b>Detalle de Auditoría</b>', S_COL_HDR),
         Paragraph('<b>Nota Final</b>', S_COL_HDR)],
        [Paragraph(f'1ª Auditoría ({id_grab})', S_INFO_L),
         Paragraph(_fmt(nota), ParagraphStyle('nv', parent=S_INFO_L,
                   alignment=TA_CENTER, fontName=FB, textColor=nc))],
    ]
    at = Table(audit_data, colWidths=[TW * 0.80, TW * 0.20])
    at.setStyle(TableStyle([
        ('BACKGROUND',    (0,0),(-1,0), C_BLUE),
        ('GRID',          (0,0),(-1,-1), 0.25, C_BORDER),
        ('VALIGN',        (0,0),(-1,-1), 'MIDDLE'),
        ('ALIGN',         (1,0),(-1,-1), 'CENTER'),
        ('TOPPADDING',    (0,0),(-1,-1), 5),
        ('BOTTOMPADDING', (0,0),(-1,-1), 5),
        ('LEFTPADDING',   (0,0),(-1,-1), 6),
    ]))
    story.append(at)
    story.append(Spacer(1, 14))

    # Estadisticas rapidas
    n_si  = sum(1 for c in crit_all if c.get('respuesta','') in ('Sí','Si','SÍ'))
    n_no  = sum(1 for c in crit_all if c.get('respuesta','') == 'No')
    n_na  = sum(1 for c in crit_all if c.get('respuesta','') == 'N/A')
    n_cko = sum(1 for c in crit_all
                if c.get('respuesta','') == 'No' and
                   (c.get('critico',False) or c.get('critica',False)))

    def _sp(txt, col):
        return Paragraph(txt, ParagraphStyle(f'sp{col}', parent=_s(f's{txt}'),
                         fontName=FB, fontSize=11, alignment=TA_CENTER, textColor=col))
    st_data = [
        [Paragraph('Criterios evaluados', S_COL_HDR),
         Paragraph('Sí', S_COL_HDR),
         Paragraph('No', S_COL_HDR),
         Paragraph('N/A', S_COL_HDR),
         Paragraph('Críticos KO', S_COL_HDR)],
        [_sp(str(n_si+n_no+n_na), C_DARK),
         _sp(str(n_si), C_SI),
         _sp(str(n_no), C_NO),
         _sp(str(n_na), C_NA),
         _sp(str(n_cko), C_NO if n_cko > 0 else C_SI)],
    ]
    cw = [TW * 0.28] + [TW * 0.18] * 4
    st = Table(st_data, colWidths=cw)
    st.setStyle(TableStyle([
        ('BACKGROUND',    (0,0),(-1,0), C_BLUE_MID),
        ('BACKGROUND',    (0,1),(-1,1), C_LGRAY),
        ('GRID',          (0,0),(-1,-1), 0.25, C_BORDER),
        ('VALIGN',        (0,0),(-1,-1), 'MIDDLE'),
        ('TOPPADDING',    (0,0),(-1,-1), 6),
        ('BOTTOMPADDING', (0,0),(-1,-1), 6),
    ]))
    story.append(st)
    story.append(PageBreak())
    return story


# PAGINA 2 — Ficha + Estructura de la llamada
def build_ficha(res):
    id_grab   = res.get('id_grabacion', 'N/D')
    nota      = res.get('nota_final', 0.0)
    fecha_lla = res.get('fecha_llamada', 'N/D')
    tipif     = res.get('tipificacion', 'N/D')
    fecha_aud = res.get('fecha_auditoria', 'N/D')
    nc        = _nota_color(nota)

    story = []
    story.append(Spacer(1, 4))
    story.append(Paragraph('Detalle de Auditoría: 1ª Auditoría', S_SUBTITLE))
    story.append(Spacer(1, 6))

    ficha_rows = [
        [Paragraph(f'<b>ID Grabación:</b> {id_grab}', S_INFO_L),
         Paragraph('<b>Nota Final:</b>',
                   ParagraphStyle('fl', parent=S_INFO_L, alignment=TA_RIGHT)),
         Paragraph(_fmt(nota),
                   ParagraphStyle('fv', parent=S_INFO_L, fontName=FB,
                                  fontSize=11, alignment=TA_RIGHT, textColor=nc))],
        [Paragraph(f'<b>Fecha de llamada:</b> {fecha_lla}',    S_INFO_L), '', ''],
        [Paragraph(f'<b>Tipificación de llamada:</b> {tipif}', S_INFO_L), '', ''],
        [Paragraph(f'<b>Fecha de auditoría:</b> {fecha_aud}',  S_INFO_L), '', ''],
    ]
    ft = Table(ficha_rows, colWidths=[TW * 0.60, TW * 0.22, TW * 0.18])
    ft.setStyle(TableStyle([
        ('SPAN',          (0,1),(2,1)),
        ('SPAN',          (0,2),(2,2)),
        ('SPAN',          (0,3),(2,3)),
        ('BACKGROUND',    (0,0),(-1,-1), C_BLUE_VL),
        ('LINEABOVE',     (0,0),(-1,0), 1.0, C_BLUE),
        ('LINEBELOW',     (0,-1),(-1,-1), 1.0, C_BLUE),
        ('VALIGN',        (0,0),(-1,-1), 'MIDDLE'),
        ('ALIGN',         (1,0),(2,0), 'RIGHT'),
        ('TOPPADDING',    (0,0),(-1,-1), 3),
        ('BOTTOMPADDING', (0,0),(-1,-1), 3),
        ('LEFTPADDING',   (0,0),(-1,-1), 6),
        ('RIGHTPADDING',  (0,0),(-1,-1), 6),
    ]))
    story.append(ft)
    story.append(Spacer(1, 8))
    return story


def build_estructura(res):
    story = build_ficha(res)
    grupos = agrupar_criterios(res.get('criterios', []))
    secs   = grupos.get('Estructura de la llamada', OrderedDict())
    if secs:
        story.append(tabla_criterios('Estructura de la llamada', secs))
    story.append(PageBreak())
    return story


# PAGINA 3 — Actitud / Comunicacion + Feedback Adicional
def build_actitud_feedback(res):
    grupos = agrupar_criterios(res.get('criterios', []))
    secs   = grupos.get('Actitud / Comunicación', OrderedDict())
    story  = []

    story.append(Spacer(1, 4))
    if secs:
        story.append(tabla_criterios('Actitud / Comunicación', secs))
    story.append(Spacer(1, 16))

    obs   = res.get('observaciones', 'Sin observaciones.')
    pftes = res.get('puntos_fuertes', [])
    ameji = res.get('areas_mejora', [])

    fbk_data = [
        [Paragraph('Feedback Adicional', S_SEC_HDR)],
        [Paragraph('Observaciones', S_FBK_BLD)],
        [Paragraph(str(obs) if obs else 'Sin observaciones.', S_FBK_TXT)],
        [Paragraph('Puntos Fuertes', S_FBK_BLD)],
        [Paragraph(_fmt_list(pftes, 'Sin puntos fuertes.'), S_FBK_TXT)],
        [Paragraph('Áreas de Mejora', S_FBK_BLD)],
        [Paragraph(_fmt_areas(ameji, 'Sin areas de mejora.'), S_FBK_TXT)],
    ]
    fbk = Table(fbk_data, colWidths=[TW])
    fbk.setStyle(TableStyle([
        ('BACKGROUND',    (0,0),(-1,0), C_BLUE),
        ('TOPPADDING',    (0,0),(-1,0), 8),
        ('BOTTOMPADDING', (0,0),(-1,0), 8),
        ('LEFTPADDING',   (0,0),(-1,0), 10),
        ('BACKGROUND',    (0,1),(-1,1), C_BLUE_L),
        ('BACKGROUND',    (0,3),(-1,3), C_BLUE_L),
        ('BACKGROUND',    (0,5),(-1,5), C_BLUE_L),
        ('TOPPADDING',    (0,1),(-1,1), 5),
        ('BOTTOMPADDING', (0,1),(-1,1), 5),
        ('TOPPADDING',    (0,3),(-1,3), 5),
        ('BOTTOMPADDING', (0,3),(-1,3), 5),
        ('TOPPADDING',    (0,5),(-1,5), 5),
        ('BOTTOMPADDING', (0,5),(-1,5), 5),
        ('LEFTPADDING',   (0,1),(-1,5), 10),
        ('TOPPADDING',    (0,2),(-1,2), 7),
        ('BOTTOMPADDING', (0,2),(-1,2), 7),
        ('TOPPADDING',    (0,4),(-1,4), 7),
        ('BOTTOMPADDING', (0,4),(-1,4), 7),
        ('TOPPADDING',    (0,6),(-1,6), 7),
        ('BOTTOMPADDING', (0,6),(-1,6), 7),
        ('LEFTPADDING',   (0,2),(-1,6), 12),
        ('RIGHTPADDING',  (0,0),(-1,-1), 10),
        ('BOX',           (0,0),(-1,-1), 0.8, C_BLUE),
        ('LINEABOVE',     (0,3),(-1,3), 0.3, C_BORDER),
        ('LINEABOVE',     (0,5),(-1,5), 0.3, C_BORDER),
        ('VALIGN',        (0,0),(-1,-1), 'TOP'),
        ('FONT',          (0,0),(-1,-1), FN, 9),
    ]))
    story.append(fbk)
    return story

print('DIGICanvas + build_resumen/estructura/actitud_feedback listos')


## NODO 12e — Generar y guardar PDF

In [ ]:
story = []
story += build_resumen(resultado)
story += build_estructura(resultado)
story += build_actitud_feedback(resultado)

HEADER_H = 34
FOOTER_H = 22

frame = Frame(
    MARGIN,
    FOOTER_H + 5,
    TW,
    H - HEADER_H - 5 - FOOTER_H - 5,
    id='main',
    leftPadding=0, rightPadding=0,
    topPadding=0,  bottomPadding=0,
)

doc = BaseDocTemplate(
    str(PDF_PATH), pagesize=A4,
    leftMargin=MARGIN,  rightMargin=MARGIN,
    topMargin=HEADER_H + 5,
    bottomMargin=FOOTER_H + 5,
)
doc.addPageTemplates([PageTemplate(id='digi', frames=[frame])])

def _make_canvas(filename, *args, **kwargs):
    kwargs['resultado'] = resultado
    return DIGICanvas(filename, *args, **kwargs)

print('Generando PDF...')
doc.build(story, canvasmaker=_make_canvas)

kb = PDF_PATH.stat().st_size / 1024
print(f'PDF generado: {PDF_PATH.name}  ({kb:.1f} KB)')
print()
print('Descargar desde Colab:')
print('  from google.colab import files')
print(f'  files.download("{PDF_PATH}")')


## NODO 13 — Registro en PostgreSQL

> INSERT ... ON CONFLICT (id_grabacion) DO UPDATE para ser idempotente.
> Configura PG_* con tus credenciales antes de ejecutar.


In [ ]:
import psycopg2
from psycopg2.extras import Json

PG_HOST = 'localhost'
PG_PORT = 5432
PG_DB   = 'auditoria_digi'
PG_USER = 'postgres'
PG_PASS = 'password'

DDL = (
    "CREATE TABLE IF NOT EXISTS auditorias ("
    "    id               SERIAL PRIMARY KEY,"
    "    id_grabacion     VARCHAR(255) UNIQUE NOT NULL,"
    "    asesor           VARCHAR(100),"
    "    fecha_llamada    VARCHAR(50),"
    "    tipificacion     VARCHAR(200),"
    "    fecha_auditoria  VARCHAR(50),"
    "    servicio         VARCHAR(50),"
    "    nota_final       NUMERIC(5,2),"
    "    nota_estructura  NUMERIC(5,2),"
    "    nota_actitud     NUMERIC(5,2),"
    "    n_si             INT,"
    "    n_no             INT,"
    "    n_na             INT,"
    "    n_criticos_ko    INT,"
    "    criterios        JSONB,"
    "    observaciones    TEXT,"
    "    puntos_fuertes   JSONB,"
    "    areas_mejora     JSONB,"
    "    pdf_path         TEXT,"
    "    creado_en        TIMESTAMP DEFAULT NOW(),"
    "    actualizado_en   TIMESTAMP DEFAULT NOW()"
    ");"
)

INSERT = (
    "INSERT INTO auditorias ("
    "    id_grabacion, asesor, fecha_llamada, tipificacion,"
    "    fecha_auditoria, servicio, nota_final, nota_estructura, nota_actitud,"
    "    n_si, n_no, n_na, n_criticos_ko,"
    "    criterios, observaciones, puntos_fuertes, areas_mejora, pdf_path"
    ") VALUES ("
    "    %(id_grabacion)s, %(asesor)s, %(fecha_llamada)s, %(tipificacion)s,"
    "    %(fecha_auditoria)s, %(servicio)s, %(nota_final)s, %(nota_estructura)s, %(nota_actitud)s,"
    "    %(n_si)s, %(n_no)s, %(n_na)s, %(n_criticos_ko)s,"
    "    %(criterios)s, %(observaciones)s, %(puntos_fuertes)s, %(areas_mejora)s, %(pdf_path)s"
    ") ON CONFLICT (id_grabacion) DO UPDATE SET"
    "    nota_final      = EXCLUDED.nota_final,"
    "    nota_estructura = EXCLUDED.nota_estructura,"
    "    nota_actitud    = EXCLUDED.nota_actitud,"
    "    criterios       = EXCLUDED.criterios,"
    "    observaciones   = EXCLUDED.observaciones,"
    "    puntos_fuertes  = EXCLUDED.puntos_fuertes,"
    "    areas_mejora    = EXCLUDED.areas_mejora,"
    "    pdf_path        = EXCLUDED.pdf_path,"
    "    actualizado_en  = NOW()"
    " RETURNING id;"
)

def insertar_resultado(res, pdf_path_str):
    crit_all = res.get('criterios', [])
    n_si  = sum(1 for c in crit_all if c.get('respuesta','') in ('Sí','Si','SÍ'))
    n_no  = sum(1 for c in crit_all if c.get('respuesta','') == 'No')
    n_na  = sum(1 for c in crit_all if c.get('respuesta','') == 'N/A')
    n_cko = sum(1 for c in crit_all
                if c.get('respuesta','') == 'No' and
                   (c.get('critico', False) or c.get('critica', False)))
    params = {
        'id_grabacion':    res.get('id_grabacion', 'UNKNOWN'),
        'asesor':          res.get('asesor', ''),
        'fecha_llamada':   res.get('fecha_llamada', ''),
        'tipificacion':    res.get('tipificacion', ''),
        'fecha_auditoria': res.get('fecha_auditoria', ''),
        'servicio':        res.get('servicio', 'C2C'),
        'nota_final':      res.get('nota_final', 0.0),
        'nota_estructura': res.get('nota_estructura', None),
        'nota_actitud':    res.get('nota_actitud', None),
        'n_si':            n_si,
        'n_no':            n_no,
        'n_na':            n_na,
        'n_criticos_ko':   n_cko,
        'criterios':       Json(crit_all),
        'observaciones':   res.get('observaciones', ''),
        'puntos_fuertes':  Json(res.get('puntos_fuertes', [])),
        'areas_mejora':    Json(res.get('areas_mejora', [])),
        'pdf_path':        pdf_path_str,
    }
    try:
        conn = psycopg2.connect(
            host=PG_HOST, port=PG_PORT, dbname=PG_DB,
            user=PG_USER, password=PG_PASS
        )
        with conn:
            with conn.cursor() as cur:
                cur.execute(DDL)
                cur.execute(INSERT, params)
                row_id = cur.fetchone()[0]
        conn.close()
        print(f'PostgreSQL: auditoria guardada con id={row_id}')
        return row_id
    except Exception as e:
        print(f'PostgreSQL no disponible (configura PG_*): {e}')
        return None

insertar_resultado(resultado, str(PDF_PATH))


## Verificacion final

In [ ]:
print('=' * 60)
print('RESUMEN FASE 4')
print('=' * 60)

if PDF_PATH.exists():
    kb = PDF_PATH.stat().st_size / 1024
    print(f'PDF  : {PDF_PATH.name}  ({kb:.1f} KB)')
else:
    print('PDF  : NO generado')

crit_all = resultado.get('criterios', [])
n_si  = sum(1 for c in crit_all if c.get('respuesta','') in ('Sí','Si','SÍ'))
n_no  = sum(1 for c in crit_all if c.get('respuesta','') == 'No')
n_na  = sum(1 for c in crit_all if c.get('respuesta','') == 'N/A')

print()
print(f'Nota final     : {resultado.get("nota_final",0):.2f}/10')
print(f'Criterios Si   : {n_si}')
print(f'Criterios No   : {n_no}')
print(f'Criterios N/A  : {n_na}')
print()
print('Estructura del informe:')
print('  Pagina 1 -> Resumen de Rendimiento')
print('  Pagina 2 -> Estructura de la llamada')
print('  Pagina 3 -> Actitud / Comunicacion + Feedback Adicional')
print()
print('Fase 4 completa.')
